# EB-NeRD — iteration 1  (CPU session)

Attach your `ebnerd_large_dataset` and run top to bottom. No GPU: Ekstra Bladet supplies
pre-computed article embeddings, which this notebook downloads from their public S3 bucket.

**This is not a copy of the MIND notebook.** EB-NeRD differs in ways that would break a
straight port, and several of them are silent failures rather than errors:

| | MIND | EB-NeRD |
|---|---|---|
| Submission filename | `prediction.txt` | **`predictions.txt`** (plural) |
| Daily submissions | 10 | **5** |
| Format | parquet? no — TSV | Parquet |
| History | inline on every impression row | **separate file**, one row per user |
| Labels | glued to the candidate (`N123-1`) | **separate list**, must be intersected |
| Publication time | absent (we reconstructed a proxy) | **supplied and exact** |
| Popularity stats | none | **`total_pageviews` / `total_inviews` supplied** |
| Article embeddings | we had to encode them | **supplied as artifacts** |

**What we already measured on the demo bundle**, which drives the feature choices below:

- `total_pageviews` alone scores **AUC 0.5969** — the strongest single feature available.
- Article age is **strongly non-monotonic**: click rate peaks at 2–4 h and is *lower* under
  1 h, with an **8.1x** spread between best and worst bucket. Yet a linear `-age` term
  scores **0.4950 — below random.** Exactly the trap that cost us a feature on MIND, but
  with a bigger prize here, so age enters as a bucketed lookup.
- `sentiment_score` scores 0.5054, i.e. nothing. Included only as an ablation row.

## 1. Setup, data discovery, and the embedding artifact

In [1]:
import gc, glob, json, re, time, warnings, zipfile, urllib.request
from collections import defaultdict, Counter
from pathlib import Path

import numpy as np
import polars as pl
import scipy.sparse as sp

import os, shutil, psutil

# A2 NRMS dependencies
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(30)
WORK = Path("/kaggle/working")
_PROC = psutil.Process(os.getpid())


def mem(tag=""):
    """Resident memory and free disk. Kaggle gives ~30 GB RAM on CPU but only ~13 GB on a
    P100 GPU session; /kaggle/working is separately capped, and the first full run died of
    ENOSPC rather than OOM, so both numbers are worth watching."""
    print(f"   [mem] {_PROC.memory_info().rss/1e9:5.2f} GB | "
          f"{shutil.disk_usage(WORK).free/1e9:5.1f} GB free on {WORK}   {tag}")


def _free(p):
    try:
        return shutil.disk_usage(p).free
    except OSError:
        return -1


# predictions.txt runs to several hundred MB and /kaggle/working is the tightest mount,
# so the big intermediate goes wherever there is actually room; only the zip must land
# in /kaggle/working to be downloadable.
SCRATCH = max([WORK, Path("/tmp"), Path("/kaggle/temp")], key=_free)


print("polars", pl.__version__)
print(f"RAM available to this session: {psutil.virtual_memory().total/1e9:.0f} GB")
for _p in (WORK, Path("/tmp"), Path("/kaggle/temp")):
    if _free(_p) >= 0:
        print(f"   {str(_p):<16} {_free(_p)/1e9:6.1f} GB free")
print(f"scratch for large intermediates: {SCRATCH}")

# /kaggle/working is a small mount that survives between runs of the same notebook, and
# this notebook used to write ~1.5 GB into it. Clear only the files it creates itself,
# then list whatever else is in there -- a full working dir is what ENOSPC actually means.
for _p in ("contrastive.zip", "predictions.txt", "ebnerd_predictions.zip"):
    if (WORK / _p).exists():
        print(f"   removing stale {_p} ({(WORK / _p).stat().st_size/1e6:.0f} MB)")
        (WORK / _p).unlink()
if (WORK / "Ekstra_Bladet_contrastive_vector").exists():
    print("   removing stale Ekstra_Bladet_contrastive_vector/")
    shutil.rmtree(WORK / "Ekstra_Bladet_contrastive_vector", ignore_errors=True)

_left = [q for q in WORK.rglob("*") if q.is_file()]
print(f"{WORK} now holds {sum(q.stat().st_size for q in _left)/1e9:.2f} GB "
      f"in {len(_left)} files, {_free(WORK)/1e9:.1f} GB free")
for q in sorted(_left, key=lambda x: -x.stat().st_size)[:8]:
    print(f"   {q.stat().st_size/1e6:9.1f} MB  {q.relative_to(WORK)}")
mem("startup")


def find_dir(*must_contain):
    """Locate a directory under /kaggle/input holding all the named files."""
    for cand in glob.glob("/kaggle/input/**/", recursive=True):
        if all((Path(cand) / f).exists() for f in must_contain):
            return Path(cand)
    raise FileNotFoundError(f"no directory contains {must_contain}")


TRAIN_DIR = find_dir("behaviors.parquet", "history.parquet")
LARGE = None
for cand in glob.glob("/kaggle/input/**/ebnerd_large/", recursive=True):
    LARGE = Path(cand); break
TEST = None
for cand in glob.glob("/kaggle/input/**/ebnerd_testset/", recursive=True):
    if (Path(cand) / "test").exists():
        TEST = Path(cand); break
    inner = Path(cand) / "ebnerd_testset"
    if (inner / "test").exists():
        TEST = inner; break

assert LARGE is not None, "could not find ebnerd_large/"
assert TEST is not None, "could not find ebnerd_testset/ with a test/ subfolder"
print("large:", LARGE)
print("test: ", TEST)
for p in [LARGE / "articles.parquet", LARGE / "train/behaviors.parquet",
          LARGE / "validation/behaviors.parquet", TEST / "test/behaviors.parquet",
          TEST / "test/history.parquet"]:
    print(f"   {'OK ' if p.exists() else 'MISSING'} {p.name:<20} {p.parent.name}")

polars 1.35.2
RAM available to this session: 34 GB
   /kaggle/working    20.9 GB free
   /tmp             1100.2 GB free
scratch for large intermediates: /tmp
/kaggle/working now holds 0.00 GB in 1 files, 20.9 GB free
         0.1 MB  .virtual_documents/__notebook_source__.ipynb
   [mem]  0.63 GB |  20.9 GB free on /kaggle/working   startup
large: /kaggle/input/datasets/suyashnpande/ebnerd-large-dataset2/ebnerd_large
test:  /kaggle/input/datasets/suyashnpande/ebnerd-large-dataset2/ebnerd_testset/ebnerd_testset
   OK  articles.parquet     ebnerd_large
   OK  behaviors.parquet    train
   OK  behaviors.parquet    validation
   OK  behaviors.parquet    test
   OK  history.parquet      test


### The article embeddings

Ekstra Bladet's own contrastively-trained model, built on this exact Danish corpus. The
bucket is public, so we pull it straight in rather than uploading anything — and cache it
to `/kaggle/working` so a re-run inside the same session is instant.

In [2]:
ART_URL = "https://ebnerd-dataset.s3.eu-west-1.amazonaws.com/artifacts/Ekstra_Bladet_contrastive_vector.zip"
ART_ZIP = SCRATCH / "contrastive.zip"

if not ART_ZIP.exists():
    t0 = time.time()
    print("downloading the contrastive embedding artifact (~358 MB) ...")
    urllib.request.urlretrieve(ART_URL, ART_ZIP)
    print(f"   done in {time.time()-t0:.0f}s ({ART_ZIP.stat().st_size/1e6:.0f} MB)")

with zipfile.ZipFile(ART_ZIP) as z:
    members = [n for n in z.namelist()
               if n.endswith(".parquet") and not n.startswith("__MACOSX")]
    print("archive parquet files:", members)
    z.extractall(SCRATCH, members=members)

EMB_PATH = SCRATCH / members[0]
emb = pl.read_parquet(EMB_PATH)
print(f"\nembedding table: {emb.height:,} rows, columns {emb.columns}")
VEC_COL = [c for c in emb.columns if c != "article_id"][0]
DIM = len(emb[VEC_COL][0])
print(f"vector column '{VEC_COL}', dimension {DIM}")

downloading the contrastive embedding artifact (~358 MB) ...
   done in 5s (358 MB)
archive parquet files: ['Ekstra_Bladet_contrastive_vector/contrastive_vector.parquet']

embedding table: 125,541 rows, columns ['article_id', 'contrastive_vector']
vector column 'contrastive_vector', dimension 768


## 2. Metrics — official definitions plus a verified fast path

In [3]:
from sklearn.metrics import roc_auc_score


def dcg_score(y_true, y_score, k=10):
    order = np.argsort(y_score)[::-1]
    y_true = np.take(y_true, order[:k])
    return np.sum((2 ** y_true - 1) / np.log2(np.arange(len(y_true)) + 2))


def ndcg_score(y_true, y_score, k=10):
    return dcg_score(y_true, y_score, k) / dcg_score(y_true, y_true, k)


def mrr_score(y_true, y_score):
    order = np.argsort(y_score)[::-1]
    y_true = np.take(y_true, order)
    return np.sum(y_true / (np.arange(len(y_true)) + 1)) / np.sum(y_true)


def score_official(L, S):
    a, m, n5, n10 = [], [], [], []
    for y, s in zip(L, S):
        y = np.asarray(y, dtype="float32")
        if y.sum() == 0 or y.sum() == len(y):
            continue
        s = np.asarray(s, dtype="float64")
        a.append(roc_auc_score(y, s)); m.append(mrr_score(y, s))
        n5.append(ndcg_score(y, s, 5)); n10.append(ndcg_score(y, s, 10))
    return {"AUC": np.mean(a), "MRR": np.mean(m), "nDCG@5": np.mean(n5),
            "nDCG@10": np.mean(n10), "n": len(a)}


def score_fast(imp_ids, labels, scores):
    """Every metric for every impression at once, via group-wise ranking."""
    df = pl.DataFrame({"imp": imp_ids, "y": np.asarray(labels, dtype=np.float64),
                       "s": np.asarray(scores, dtype=np.float64)})
    df = df.with_columns([
        pl.col("s").rank(method="average").over("imp").alias("r_asc"),
        pl.col("s").rank(method="ordinal", descending=True).over("imp").alias("r_desc"),
    ]).with_columns([
        (pl.col("y") * pl.col("r_asc")).alias("pos_rank"),
        (pl.col("y") / pl.col("r_desc")).alias("rr"),
        pl.when(pl.col("r_desc") <= 5).then(pl.col("y") / (pl.col("r_desc") + 1).log(2))
          .otherwise(0.0).alias("g5"),
        pl.when(pl.col("r_desc") <= 10).then(pl.col("y") / (pl.col("r_desc") + 1).log(2))
          .otherwise(0.0).alias("g10"),
    ])
    g = df.group_by("imp").agg([
        pl.len().alias("n"), pl.col("y").sum().alias("npos"),
        pl.col("pos_rank").sum().alias("rank_sum"), pl.col("rr").sum().alias("rr_sum"),
        pl.col("g5").sum().alias("dcg5"), pl.col("g10").sum().alias("dcg10"),
    ]).filter((pl.col("npos") > 0) & (pl.col("npos") < pl.col("n")))
    n = g["n"].to_numpy(); npos = g["npos"].to_numpy()
    auc = (g["rank_sum"].to_numpy() - npos * (npos + 1) / 2) / (npos * (n - npos))
    disc = 1.0 / np.log2(np.arange(2, 12))
    cum = np.concatenate([[0.0], np.cumsum(disc)])
    return {"AUC": auc.mean(), "MRR": (g["rr_sum"].to_numpy() / npos).mean(),
            "nDCG@5": (g["dcg5"].to_numpy() / cum[np.minimum(npos, 5).astype(int)]).mean(),
            "nDCG@10": (g["dcg10"].to_numpy() / cum[np.minimum(npos, 10).astype(int)]).mean(),
            "n": len(auc)}


# Verify before relying on it. AUC uses average ranks so it is tie-safe and must match
# exactly; MRR/nDCG need a strict order and the official np.argsort(...)[::-1] is not a
# stable descending sort, so under ties its ordering is arbitrary.
rng = np.random.default_rng(0)
imp, ys, ss, L, S = [], [], [], [], []
for i in range(4000):
    k = int(rng.integers(2, 25))
    y = np.zeros(k, dtype=int); y[rng.integers(0, k)] = 1     # EB-NeRD: ~1 click/impression
    s = np.round(rng.normal(size=k), 2)
    imp += [i] * k; ys += list(y); ss += list(s); L.append(y); S.append(s)
f, o = score_fast(np.array(imp), np.array(ys), np.array(ss)), score_official(L, S)
for k in ["AUC", "MRR", "nDCG@5", "nDCG@10"]:
    print(f"   {k:<9} fast {f[k]:.6f}   official {o[k]:.6f}   diff {abs(f[k]-o[k]):.2e}")
assert abs(f["AUC"] - o["AUC"]) < 1e-9, "AUC must match exactly"
assert max(abs(f[k] - o[k]) for k in ["MRR", "nDCG@5", "nDCG@10"]) < 1e-3
print("\nverified against the official definitions")

   AUC       fast 0.499516   official 0.499516   diff 5.55e-17
   MRR       fast 0.300269   official 0.300344   diff 7.55e-05
   nDCG@5    fast 0.308164   official 0.308057   diff 1.08e-04
   nDCG@10   fast 0.394901   official 0.395029   diff 1.28e-04

verified against the official definitions


## 3. Load

`history.parquet` is 1.2 GB and holds one list per user covering 21 days. We project to
just the two columns we need and truncate each list to its most recent entries **during**
the scan, so the full column never lands in memory.

In [4]:
HIST_CAP = 30          # most recent clicks kept per user

# Project at read time. The full behaviours schema also carries read_time,
# scroll_percentage, device_type, demographics and the next_* columns, none of which we
# use; at 12M+ rows per split that is several GB of nothing. Ask parquet for five columns.
BH_COLS = ["impression_id", "impression_time", "user_id",
           "article_ids_inview", "article_ids_clicked"]
# articles.parquet is 82% `body` (full article text) plus url/ner_clusters/topics/
# image_ids, none of which we read. Nine columns instead of seventeen.
ART_COLS = ["article_id", "published_time", "category_str", "subcategory",
            "total_pageviews", "total_inviews", "sentiment_score", "title", "subtitle"]

def read_history(path):
    return (pl.scan_parquet(path)
              .select([pl.col("user_id"),
                       pl.col("article_id_fixed").list.tail(HIST_CAP).alias("hist")])
              .collect(streaming=True))


t0 = time.time()
articles = pl.read_parquet(LARGE / "articles.parquet", columns=ART_COLS)
bh_train = pl.read_parquet(LARGE / "train/behaviors.parquet", columns=BH_COLS)
bh_val = pl.read_parquet(LARGE / "validation/behaviors.parquet", columns=BH_COLS)
bh_test = pl.read_parquet(TEST / "test/behaviors.parquet",
                          columns=[c for c in BH_COLS if c != "article_ids_clicked"])
print(f"behaviors loaded in {time.time()-t0:.0f}s")

# Each split ships its own 21-day lookback, ending exactly where that split's behaviours
# begin. Pooling them and keeping one row per user takes the LAST file -- the test
# history, which runs to the start of the test week and therefore covers the whole
# validation week we tune on. That put the labels being predicted inside the user
# profile. Keep them separate: validation history to evaluate and tune, test history to
# predict. Same code path, different lookback.
t0 = time.time()
HIST_TRAIN = read_history(LARGE / "train/history.parquet")
HIST_EVAL = read_history(LARGE / "validation/history.parquet")
HIST_PRED = read_history(TEST / "test/history.parquet")
print(f"history: {HIST_EVAL.height:,} eval users | {HIST_PRED.height:,} pred users "
      f"in {time.time()-t0:.0f}s")

print(f"\ntrain {bh_train.height:,} | validation {bh_val.height:,} | test {bh_test.height:,}")
print(f"articles {articles.height:,}")
TEST_SCHEMA = pl.scan_parquet(TEST / "test/behaviors.parquet").collect_schema().names()
print("\ntest columns:", TEST_SCHEMA)
HAS_BA = "is_beyond_accuracy" in TEST_SCHEMA
print(f"is_beyond_accuracy present: {HAS_BA}")
for nm, df in [("train", bh_train), ("validation", bh_val), ("test", bh_test)]:
    print(f"   {nm:<11} {df['impression_time'].min()}  ->  {df['impression_time'].max()}")

TEST_ROWS = int(bh_test["article_ids_inview"].list.len().sum())
print(f"\ntest candidate rows: {TEST_ROWS:,} "
      f"({TEST_ROWS/bh_test.height:.1f} per impression) <- what section 10 has to score")

behaviors loaded in 12s
history: 791,582 eval users | 807,677 pred users in 9s

train 12,063,890 | validation 12,566,385 | test 13,536,710
articles 125,541

test columns: ['impression_id', 'impression_time', 'read_time', 'scroll_percentage', 'device_type', 'article_ids_inview', 'user_id', 'is_sso_user', 'gender', 'postcode', 'age', 'is_subscriber', 'session_id', 'is_beyond_accuracy']
is_beyond_accuracy present: True
   train       2023-05-18 07:00:00  ->  2023-05-25 06:59:59
   validation  2023-05-25 07:00:00  ->  2023-06-01 06:59:59
   test        2023-06-01 07:00:00  ->  2023-06-08 06:59:59

test candidate rows: 205,925,868 (15.2 per impression) <- what section 10 has to score


## 4. Article features

Four signals, three of which MIND simply did not have.

`total_pageviews` is the strongest single feature we measured (AUC 0.5969 alone). One
caveat we record rather than hide: it counts clicks in the **first 7 days after
publication**, so for an article published during the test week it partly reflects the
future. It is supplied in the dataset and therefore fair game for the leaderboard, but it
is not something a live system could read at serving time — exactly the Q9
"features unavailable at serving time" category, and the ablation below reports the score
with and without it.

In [5]:
# Age buckets measured on the demo bundle: click rate peaks at 2-4h and falls away on
# BOTH sides, so a linear term cannot express it (it scored 0.4950, below random).
AGE_EDGES = np.array([1, 2, 4, 8, 12, 18, 24, 48, 120], dtype=np.float64)
AGE_VALUE = np.array([0.0773, 0.1090, 0.1193, 0.1112, 0.1108,
                      0.0884, 0.0764, 0.0445, 0.0394, 0.0146])

ART = articles.select([
    pl.col("article_id"),
    pl.col("published_time"),
    pl.col("category_str").fill_null("").alias("category"),
    pl.col("subcategory").list.first().fill_null(-1).alias("subcat"),
    pl.col("total_pageviews").fill_null(0).alias("pageviews"),
    pl.col("total_inviews").fill_null(0).alias("inviews"),
    pl.col("sentiment_score").fill_null(0.0).alias("sentiment"),
    pl.col("title").fill_null(""),
    pl.col("subtitle").fill_null(""),
]).sort("article_id")

CATALOGUE = ART["article_id"].to_list()
IDX = {int(a): i for i, a in enumerate(CATALOGUE)}
ART_IDS = np.asarray(CATALOGUE, dtype=np.int64)    # sorted: ART was sorted by article_id
ID_DTYPE = ART.schema["article_id"]
print(f"catalogue {len(CATALOGUE):,} articles")


def lookup(sorted_keys, values):
    """Vectorised dict.get over a sorted key array -> (positions, found mask)."""
    v = np.asarray(values, dtype=np.int64)
    p = np.searchsorted(sorted_keys, v)
    np.clip(p, 0, len(sorted_keys) - 1, out=p)
    return p, sorted_keys[p] == v

# log1p because pageviews spans several orders of magnitude
ART = ART.with_columns([
    (pl.col("pageviews") + 1).log().alias("log_pv"),
    (pl.col("inviews") + 1).log().alias("log_iv"),
])
print(f"pageviews non-zero on {100*(ART['pageviews']>0).mean():.1f}% of articles")

# Embeddings, re-indexed onto the catalogue order so a row index means the same thing
# everywhere. Articles without a vector keep zeros, which score 0 against everything.
VEC = np.zeros((len(CATALOGUE), DIM), dtype=np.float32)
src = {int(a): i for i, a in enumerate(emb["article_id"].to_list())}
# explode to one flat buffer rather than np.stack over 125k little object arrays
raw = (emb[VEC_COL].explode().to_numpy()
       .astype(np.float32, copy=False).reshape(emb.height, DIM))
missing = 0
for a, i in IDX.items():
    j = src.get(a)
    if j is None:
        missing += 1
    else:
        VEC[i] = raw[j]
n = np.linalg.norm(VEC, axis=1, keepdims=True)
VEC /= np.maximum(n, 1e-12)                    # unit vectors => dot product is cosine
del raw, src, emb
# ~716 MB of zip plus extracted parquet, now fully absorbed into VEC
shutil.rmtree(SCRATCH / "Ekstra_Bladet_contrastive_vector", ignore_errors=True)
ART_ZIP.unlink(missing_ok=True)
gc.collect()
print(f"embeddings: {missing:,} of {len(CATALOGUE):,} articles without a vector "
      f"({100*missing/len(CATALOGUE):.2f}%), matrix {VEC.nbytes/1e6:.0f} MB")
mem("after embeddings")

catalogue 125,541 articles
pageviews non-zero on 13.5% of articles
embeddings: 0 of 125,541 articles without a vector (0.00%), matrix 386 MB
   [mem]  6.72 GB |  20.9 GB free on /kaggle/working   after embeddings


## 5. BM25 over Danish text

Same inverted index as MIND, but with Danish stopwords. Danish is a compounding language —
`fodboldlandsholdet` is one token meaning "the national football team" — so exact term
matching is structurally handicapped here. We include BM25 anyway because it completes Q2
at scale and because the MIND run showed it earns a non-zero weight even when semantic
dominates.

In [6]:
TOKEN = re.compile(r"[^\W_]+", re.UNICODE)
STOP_DA = set(("og i jeg det at en den til er som på de med han af for ikke der var mig "
               "sig men et har om vi min havde ham hun nu over da fra du ud sin dem os "
               "op man hans hvor eller hvad skal selv her alle vil blev kunne ind når "
               "være dog no mit efter" ).split())
K1, B_ = 1.2, 0.75


def tok(s):
    return [w for w in TOKEN.findall(s.lower()) if len(w) > 2 and w not in STOP_DA]


t0 = time.time()
docs = [tok(t + " " + s) for t, s in zip(ART["title"].to_list(), ART["subtitle"].to_list())]
vocab, postings = {}, []
doc_len = np.zeros(len(docs), dtype=np.float32)
for d, ts in enumerate(docs):
    doc_len[d] = len(ts)
    cnt = defaultdict(int)
    for w in ts:
        t = vocab.get(w)
        if t is None:
            t = len(vocab); vocab[w] = t; postings.append({})
        cnt[t] += 1
    for t, tf in cnt.items():
        postings[t][d] = tf

avgdl = float(doc_len.mean()); N = len(docs)
nnz = sum(len(p) for p in postings)
rows = np.empty(nnz, np.int32); cols = np.empty(nnz, np.int32)
data = np.empty(nnz, np.float32); tfs = np.empty(nnz, np.float32)
pos = 0
for t, p in enumerate(postings):
    if not p:
        continue
    idf = np.log(1.0 + (N - len(p) + 0.5) / (len(p) + 0.5))
    for d, tf in p.items():
        norm = K1 * (1.0 - B_ + B_ * doc_len[d] / avgdl)
        rows[pos] = d; cols[pos] = t; data[pos] = idf * tf * (K1 + 1.0) / (tf + norm)
        tfs[pos] = tf
        pos += 1
BM25_W = sp.csr_matrix((data[:pos], (rows[:pos], cols[:pos])), shape=(N, len(vocab)))
# The raw counts as well. A user's BM25 query is the summed term counts of the articles
# they clicked, which is exactly (click matrix @ TF) -- see section 7.
TF = sp.csr_matrix((tfs[:pos], (rows[:pos], cols[:pos])), shape=(N, len(vocab)))
del postings, rows, cols, data, tfs, docs
gc.collect()
print(f"BM25: {N:,} docs, {len(vocab):,} terms, {BM25_W.nnz:,} postings "
      f"in {time.time()-t0:.0f}s")

BM25: 125,541 docs, 119,851 terms, 1,774,642 postings in 8s


## 6. Click-through rate, built to a cutoff

Same idea that worked on MIND: an exponential half-life so recent clicks count for more,
and the ability to build the table to an arbitrary cutoff so we can tune under the same
staleness the test set will have.

In [7]:
HALF_LIFE_H = 24.0
ALPHA = 20.0
CTR_STEP = 500_000
# A list of projections rather than pl.concat: selecting columns is a refcount bump on the
# Arrow buffers, whereas concatenating 24.6M rows would copy ~1.5 GB for no benefit.
_LAB_COLS = ["impression_time", "article_ids_inview", "article_ids_clicked"]
LABELLED = [bh_train.select(_LAB_COLS), bh_val.select(_LAB_COLS)]


def build_ctr(cutoff):
    """Decay-weighted click-through rate per article, accumulated in fixed-size slices.

    Exploding every labelled impression at once is ~296M rows and several GB in a single
    intermediate, and this runs three times. Accumulating into two arrays over slices
    keeps the transient flat no matter how much history the cutoff admits.
    """
    assert cutoff is not None, "null cutoff: check how it was derived"
    parts = [p.filter(pl.col("impression_time") < cutoff) for p in LABELLED]
    parts = [p for p in parts if p.height]
    assert parts, f"no labelled impressions before {cutoff}"
    ref = max(p["impression_time"].max() for p in parts)

    shown = np.zeros(len(CATALOGUE), np.float64)
    clicks = np.zeros(len(CATALOGUE), np.float64)
    for part in parts:
        for st in range(0, part.height, CTR_STEP):
            ex = (part.slice(st, CTR_STEP)
                      .with_columns(pl.col("article_ids_inview").alias("a")).explode("a")
                      .with_columns([
                          pl.col("a").is_in(pl.col("article_ids_clicked"))
                            .cast(pl.Int8).alias("y"),
                          (0.5 ** (((pl.lit(ref) - pl.col("impression_time"))
                                    .dt.total_seconds() / 3600.0) / HALF_LIFE_H)).alias("w"),
                      ]).select(["a", "y", "w"]))
            ai, hit = lookup(ART_IDS, ex["a"].to_numpy())
            w = ex["w"].to_numpy()[hit]
            ai = ai[hit]
            shown += np.bincount(ai, weights=w, minlength=len(CATALOGUE))
            clicks += np.bincount(ai, weights=w * ex["y"].to_numpy()[hit],
                                  minlength=len(CATALOGUE))
            del ex
    g = float(clicks.sum() / max(shown.sum(), 1e-12))
    keep = shown > 0
    return pl.DataFrame({
        "article_id": ART_IDS[keep],
        "ctr": (clicks[keep] + ALPHA * g) / (shown[keep] + ALPHA),
    }).with_columns(pl.col("article_id").cast(ID_DTYPE)), g


# The test week follows validation, so "CTR from train, evaluated on validation" already
# reproduces the staleness the test set will see. We build a second, staler table to show
# the effect explicitly and to tune on the harder of the two.
# NB: Series.quantile() on a datetime column returns null on polars < ~1.40 (Kaggle ships
# 1.35), which silently makes the cutoff null, empties the filter, and only surfaces much
# later as "- not allowed on null and datetime". Plain datetime arithmetic instead: it is
# the true temporal midpoint and behaves the same on every version.
VAL_START = bh_val["impression_time"].min()
VAL_MID = VAL_START + (bh_val["impression_time"].max() - VAL_START) / 2
print(f"validation midpoint: {VAL_MID}")
t0 = time.time()
CTR_FRESH, G_FRESH = build_ctr(VAL_MID)                 # includes half of validation
CTR_STALE, G_STALE = build_ctr(VAL_START)               # train only
CTR_FINAL, G_FINAL = build_ctr(bh_test["impression_time"].min())   # everything labelled
print(f"fresh {CTR_FRESH.height:,} | stale {CTR_STALE.height:,} | final {CTR_FINAL.height:,} "
      f"articles ({time.time()-t0:.0f}s)")
mem("after CTR tables")

validation midpoint: 2023-05-28 18:59:59.500000
fresh 11,474 | stale 10,116 | final 12,818 articles (170s)
   [mem]  9.38 GB |  20.9 GB free on /kaggle/working   after CTR tables


### Recalibrating the age curve

`AGE_VALUE` in section 4 was measured on the demo bundle, roughly 1% of this corpus. Now
that the real logs are loaded we recompute it here rather than transplanting constants
across a 100x scale gap, and print both so the difference is on the record. Buckets with
too little support keep the demo value.

In [8]:
MIN_BUCKET = 5_000
AGE_LABELS = ["<1h", "1-2h", "2-4h", "4-8h", "8-12h", "12-18h", "18-24h", "24-48h",
              "48-120h", ">120h"]

t0 = time.time()
# Train only -- fitting this on validation would leak into the tuning we do there. And
# strided rather than head(), so the sample spans the whole training week instead of its
# first few hours, when every article in the catalogue still looks freshly published.
_step = max(1, LABELLED[0].height // 1_000_000)
_src = (LABELLED[0].gather_every(_step)
        .with_columns(pl.col("article_ids_inview").alias("a")).explode("a")
        .with_columns(pl.col("a").is_in(pl.col("article_ids_clicked")).cast(pl.Int8).alias("y"))
        .join(ART.select(["article_id", "published_time"]),
              left_on="a", right_on="article_id", how="left")
        .with_columns(((pl.col("impression_time") - pl.col("published_time"))
                       .dt.total_seconds() / 3600.0).alias("age_h"))
        .drop_nulls("age_h").filter(pl.col("age_h") >= 0))
_g = (_src.with_columns(pl.Series("b", np.searchsorted(AGE_EDGES, _src["age_h"].to_numpy())))
          .group_by("b").agg([pl.len().alias("n"), pl.col("y").mean().alias("ctr")])
          .sort("b"))

AGE_DEMO = AGE_VALUE.copy()
AGE_VALUE = AGE_VALUE.copy()
_seen = dict(zip(_g["b"].to_list(), zip(_g["n"].to_list(), _g["ctr"].to_list())))
for b, (cnt, ctr) in _seen.items():
    if cnt >= MIN_BUCKET:
        AGE_VALUE[b] = ctr

print(f"recomputed on {_src.height:,} candidate rows ({time.time()-t0:.0f}s)\n")
print(f"{'bucket':<9} {'rows':>12} {'demo':>8} {'large':>8}")
for i, lab in enumerate(AGE_LABELS):
    cnt = _seen.get(i, (0, 0.0))[0]
    flag = "" if cnt >= MIN_BUCKET else "   <- too few, kept demo"
    print(f"{lab:<9} {cnt:>12,} {AGE_DEMO[i]:>8.4f} {AGE_VALUE[i]:>8.4f}{flag}")
print(f"\nspread (max/min): demo {AGE_DEMO.max()/AGE_DEMO.min():.1f}x  "
      f"large {AGE_VALUE.max()/AGE_VALUE.min():.1f}x")
print(f"peak bucket: demo {AGE_LABELS[int(AGE_DEMO.argmax())]}, "
      f"large {AGE_LABELS[int(AGE_VALUE.argmax())]}")
del _src, _g
# LABELLED and bh_train have now done their only jobs -- the CTR tables above and this
# age curve. Together they are ~2 GB that nothing downstream reads. pop() rather than del
# so re-running the cell is not a NameError.
# Keep bh_train and HIST_TRAIN for the A2 NRMS training stage.
# LABELLED is no longer needed.
globals().pop("LABELLED", None)
gc.collect()
mem("after freeing LABELLED; bh_train retained for NRMS")

recomputed on 11,140,470 candidate rows (5s)

bucket            rows     demo    large
<1h          1,866,281   0.0773   0.0838
1-2h         1,779,294   0.1090   0.1161
2-4h         2,005,446   0.1193   0.1263
4-8h         1,421,123   0.1112   0.1292
8-12h          857,426   0.1108   0.1319
12-18h         365,322   0.0884   0.0975
18-24h         173,531   0.0764   0.0497
24-48h         475,538   0.0445   0.0334
48-120h        324,340   0.0394   0.0288
>120h        1,872,169   0.0146   0.0141

spread (max/min): demo 8.2x  large 9.4x
peak bucket: demo 2-4h, large 8-12h
   [mem] 12.07 GB |  20.9 GB free on /kaggle/working   after freeing LABELLED; bh_train retained for NRMS


## 7. Feature frame

In [9]:
ROW_CHUNK = 100_000


# ------------------------------------------------- the click matrix, and nothing derived
# Everything a user contributes -- their embedding, their BM25 query, their category mix --
# is a product against H, the user x article click matrix. H is small (~0.15 GB each). The
# products are not: materialising them for every user costs GBs, and we only ever look at
# one batch at a time, so each batch rebuilds its own slice as a sparse matmul.
#
# Two lookbacks, never pooled. Each split ships its own 21 days ending exactly where that
# split's behaviours begin; the test history runs to the start of the test week and so
# covers the whole validation week we tune on. Keeping them apart is what stops the labels
# being predicted from appearing inside the user profile.
t0 = time.time()


def click_matrix(hist):
    """(sorted user ids, user x article click matrix) for one history lookback."""
    hist = hist.sort("user_id")             # so a user's row index is searchsortable
    uid = hist["user_id"].to_numpy().astype(np.int64)
    hx = hist.with_row_index("urow").select(["urow", "hist"]).explode("hist").drop_nulls()
    ai, hit = lookup(ART_IDS, hx["hist"].to_numpy())
    H = sp.csr_matrix((np.ones(int(hit.sum()), np.float32),
                       (hx["urow"].to_numpy()[hit], ai[hit])),
                      shape=(len(uid), len(CATALOGUE)))
    return uid, H


UID_EVAL, H_EVAL = click_matrix(HIST_EVAL)
UID_PRED, H_PRED = click_matrix(HIST_PRED)
gc.collect()
print(f"click matrices in {time.time()-t0:.0f}s")
for _nm, _u, _H in [("eval (validation lookback)", UID_EVAL, H_EVAL),
                    ("pred (test lookback)      ", UID_PRED, H_PRED)]:
    print(f"   {_nm}: {len(_u):,} users, {_H.nnz:,} clicks, "
          f"{(_H.data.nbytes + _H.indices.nbytes)/1e9:.2f} GB")


def onehot(codes, width):
    return sp.csr_matrix((np.ones(len(codes), np.float32),
                          (np.arange(len(codes)), codes)), shape=(len(codes), width))


# ------------------------------------------------- article features as flat arrays
# These are constant across every batch, so joining them onto 300k rows 687 times was
# pure repetition. Held as numpy arrays indexed by catalogue position and gathered with
# the candidate index instead, which removes both hash joins from the hot loop.
A_LOGPV = ART["log_pv"].to_numpy().astype(np.float32)
A_LOGIV = ART["log_iv"].to_numpy().astype(np.float32)
A_SENT = ART["sentiment"].to_numpy().astype(np.float32)
_pub = ART["published_time"].to_numpy()
A_PUB_OK = ~np.isnat(_pub)
A_PUB = np.where(A_PUB_OK, _pub.astype("datetime64[s]").astype(np.int64), 0)
del _pub

_c = ART["category"].to_list()
_cmap = {v: i for i, v in enumerate(sorted(set(_c)))}
CAT_CODE = np.fromiter((_cmap[v] for v in _c), np.int32, len(_c))
_s = ART["subcat"].to_list()
_smap = {v: i for i, v in enumerate(sorted(set(_s)))}
SUB_CODE = np.fromiter((_smap[v] for v in _s), np.int32, len(_s))
C_CAT, C_SUB = onehot(CAT_CODE, len(_cmap)), onehot(SUB_CODE, len(_smap))
del _c, _s


def ctr_vector(tbl, g):
    """A CTR table spread onto catalogue positions, with the global rate as the default."""
    v = np.full(len(CATALOGUE), g, dtype=np.float32)
    p, hit = lookup(ART_IDS, tbl["article_id"].to_numpy())
    v[p[hit]] = tbl["ctr"].to_numpy()[hit].astype(np.float32)
    return v


CTRV_FRESH = ctr_vector(CTR_FRESH, G_FRESH)
CTRV_STALE = ctr_vector(CTR_STALE, G_STALE)
CTRV_FINAL = ctr_vector(CTR_FINAL, G_FINAL)


def prepare(df, ctr_vec, labelled):
    """Explode a behaviours slice to candidate rows, as plain numpy arrays.

    Rows come out contiguous per impression and in the original order, which is what the
    scorer requires and what lets ranking be a segmented sort rather than a group-by.
    """
    cols = ["impression_id", "user_id", "impression_time", "article_ids_inview"]
    if labelled:
        cols.append("article_ids_clicked")
    b = df.select(cols)
    imp_ids = b["impression_id"].to_numpy()
    lens = b["article_ids_inview"].list.len().to_numpy().astype(np.int64)
    ex = b.explode("article_ids_inview").rename({"article_ids_inview": "article_id"})
    if labelled:
        label = (ex.select(pl.col("article_id").is_in(pl.col("article_ids_clicked"))
                             .cast(pl.Int8).alias("y"))["y"].to_numpy())
    else:
        label = np.zeros(ex.height, dtype=np.int8)

    ci, c_ok = lookup(ART_IDS, ex["article_id"].to_numpy())
    ts = ex["impression_time"].to_numpy().astype("datetime64[s]").astype(np.int64)
    have_age = c_ok & A_PUB_OK[ci]
    age = np.where(have_age, np.maximum((ts - A_PUB[ci]) / 3600.0, 0.0), 0.0)
    z = np.zeros(ex.height, dtype=np.float32)
    return {
        "imp": np.repeat(imp_ids, lens), "imp_ids": imp_ids, "lens": lens,
        "label": label, "ci": ci, "c_ok": c_ok,
        "users": ex["user_id"].to_numpy(), "n": ex.height,
        "age_score": AGE_VALUE[np.searchsorted(AGE_EDGES, age)],
        "ctr": np.where(c_ok, ctr_vec[ci], ctr_vec.mean()),
        "pv": np.where(c_ok, A_LOGPV[ci], z), "iv": np.where(c_ok, A_LOGIV[ci], z),
        "sent": np.where(c_ok, A_SENT[ci], z),
    }


def zscore(x):
    x = np.asarray(x, dtype=np.float64); sd = x.std()
    return (x - x.mean()) / sd if sd > 1e-12 else np.zeros_like(x)


def components(F, uid, H, row_chunk=ROW_CHUNK):
    """Every scoring signal for one prepared batch.

    `uid`/`H` name the history lookback explicitly -- UID_EVAL/H_EVAL when scoring
    validation, UID_PRED/H_PRED when predicting the test week. Passing it rather than
    reading a global is the point: the two must never be mixed up.

    Peak extra memory is set by row_chunk, not by the batch or the size of the user base.
    """
    ci, n = F["ci"], F["n"]
    ui, u_ok = lookup(uid, F["users"])
    idx = np.flatnonzero(F["c_ok"] & u_ok)

    uniq, inv = np.unique(ui[idx], return_inverse=True)
    Hb = H[uniq]
    deg = np.maximum(np.asarray(Hb.sum(axis=1)).ravel(), 1.0).astype(np.float32)
    Ub = Hb @ VEC                              # sum-pool, then unit length => dot = cosine
    Ub /= np.maximum(np.linalg.norm(Ub, axis=1, keepdims=True), 1e-12)
    Qb = (Hb @ TF).tocsr()                     # BM25 query = summed term counts
    Acat = np.asarray((Hb @ C_CAT).todense(), dtype=np.float32) / deg[:, None]
    Asub = np.asarray((Hb @ C_SUB).todense(), dtype=np.float32) / deg[:, None]

    sem = np.zeros(n, np.float32); bm = np.zeros(n, np.float32)
    cat = np.zeros(n, np.float32); sub = np.zeros(n, np.float32)
    for st in range(0, len(idx), row_chunk):
        sl = idx[st:st + row_chunk]
        c, v = ci[sl], inv[st:st + row_chunk]
        sem[sl] = np.einsum("ij,ij->i", VEC[c], Ub[v])
        bm[sl] = np.asarray(BM25_W[c].multiply(Qb[v]).sum(axis=1)).ravel()
        cat[sl] = Acat[v, CAT_CODE[c]]
        sub[sl] = Asub[v, SUB_CODE[c]]
    del Hb, Ub, Qb, Acat, Asub, ui, u_ok, idx, uniq, inv

    return {"pv": zscore(F["pv"]), "iv": zscore(F["iv"]), "ctr": zscore(F["ctr"]),
            "age": zscore(F["age_score"]), "sem": zscore(sem), "bm25": zscore(bm),
            "cat": zscore(cat), "sub": zscore(sub), "sent": zscore(F["sent"])}


def segment_ranks(scores, lens):
    """Ordinal descending rank within each impression, ties broken by slate position.

    Exploded rows are already contiguous per impression, so one stable lexsort does what
    a sort + rank().over() + group_by was doing, without leaving numpy.
    """
    seg = np.repeat(np.arange(len(lens)), lens)
    order = np.lexsort((-np.asarray(scores, dtype=np.float64), seg))
    ranks = np.empty(len(seg), dtype=np.int32)
    ranks[order] = (np.arange(len(seg)) - np.repeat(np.concatenate([[0], np.cumsum(lens)[:-1]]),
                                                    lens) + 1)
    return ranks

click matrices in 6s
   eval (validation lookback): 791,582 users, 18,371,402 clicks, 0.15 GB
   pred (test lookback)      : 807,677 users, 18,728,286 clicks, 0.15 GB


## 8. Two evaluation regimes, and the single-feature ablation

FRESH uses a CTR table that includes half of validation; STALE uses train only. Both are
scored on the second half of validation. The gap between them tells us how much each
signal decays — which on MIND turned out to be the difference between a good submission
and a mediocre one.

In [10]:
EVAL_N = 60_000
val_late = bh_val.filter(pl.col("impression_time") >= VAL_MID).head(EVAL_N)
print(f"evaluating on {val_late.height:,} impressions from the second half of validation")

t0 = time.time()
FRAMES, COMP = {}, {}
for regime, ctrv in [("FRESH", CTRV_FRESH), ("STALE", CTRV_STALE)]:
    FRAMES[regime] = prepare(val_late, ctrv, True)
    COMP[regime] = components(FRAMES[regime], UID_EVAL, H_EVAL)
    print(f"   {regime}: {FRAMES[regime]['n']:,} rows ({time.time()-t0:.0f}s)")
    mem(f"after {regime} components")


def evaluate(F, score):
    return score_fast(F["imp"], F["label"], score)


rows = []
for regime in ["FRESH", "STALE"]:
    f, c = FRAMES[regime], COMP[regime]
    singles = {"random": np.random.default_rng(0).random(f["n"]),
               "total_pageviews": c["pv"], "total_inviews": c["iv"],
               "decayed CTR": c["ctr"], "age bucket": c["age"],
               "semantic (contrastive)": c["sem"], "BM25": c["bm25"],
               "category affinity": c["cat"], "subcategory affinity": c["sub"],
               "sentiment": c["sent"]}
    for name, s in singles.items():
        m = evaluate(f, s); m["regime"] = regime; m["ranker"] = name
        rows.append(m)

board = (pl.DataFrame(rows).select(["regime", "ranker", "AUC", "MRR", "nDCG@5", "nDCG@10", "n"])
           .sort(["regime", "AUC"], descending=[False, True]))
print(board)

evaluating on 60,000 impressions from the second half of validation
   FRESH: 736,204 rows (5s)
   [mem] 12.51 GB |  20.9 GB free on /kaggle/working   after FRESH components
   STALE: 736,204 rows (9s)
   [mem] 12.60 GB |  20.9 GB free on /kaggle/working   after STALE components
shape: (20, 7)
┌────────┬────────────────────────┬──────────┬──────────┬──────────┬──────────┬───────┐
│ regime ┆ ranker                 ┆ AUC      ┆ MRR      ┆ nDCG@5   ┆ nDCG@10  ┆ n     │
│ ---    ┆ ---                    ┆ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---   │
│ str    ┆ str                    ┆ f64      ┆ f64      ┆ f64      ┆ f64      ┆ i64   │
╞════════╪════════════════════════╪══════════╪══════════╪══════════╪══════════╪═══════╡
│ FRESH  ┆ age bucket             ┆ 0.670567 ┆ 0.433936 ┆ 0.493644 ┆ 0.545081 ┆ 60000 │
│ FRESH  ┆ total_pageviews        ┆ 0.611923 ┆ 0.390998 ┆ 0.436676 ┆ 0.506391 ┆ 60000 │
│ FRESH  ┆ decayed CTR            ┆ 0.572934 ┆ 0.347618 ┆ 0.392148 ┆ 0.46799  ┆ 60000 │
│

## 9. Tune the blend on the STALE regime

Two searches. The first uses everything; the second excludes `total_pageviews`, because it
partly encodes the future (clicks in the 7 days after publication) and Q9 asks for metrics
with and without features unavailable at serving time. We submit the first and report both.

In [11]:
from itertools import product

def blend(c, w):
    return sum(w[k] * c[k] for k in w)


def search(grid, comp, frame, label):
    keys = list(grid)
    best = None
    for vals in product(*(grid[k] for k in keys)):
        w = dict(zip(keys, vals))
        if all(v == 0 for v in w.values()):
            continue
        auc = evaluate(frame, blend(comp, w))["AUC"]
        if best is None or auc > best[0]:
            best = (auc, w)
    print(f"{label}: AUC {best[0]:.4f}  {best[1]}")
    return best


FULL = {"pv": [0.0, 1.0, 2.0, 3.0], "ctr": [0.0, 0.5, 1.0],
        "age": [0.0, 1.0, 2.0, 3.0], "sem": [0.0, 1.0, 2.0, 3.0],
        "bm25": [0.0, 0.5], "sub": [0.0, 0.5, 1.0]}
t0 = time.time()
BEST_AUC, BEST_W = search(FULL, COMP["STALE"], FRAMES["STALE"], "with pageviews   ")

HONEST = dict(FULL); HONEST["pv"] = [0.0]
HON_AUC, HON_W = search(HONEST, COMP["STALE"], FRAMES["STALE"], "without pageviews")
print(f"\nsearched in {time.time()-t0:.0f}s")
print(f"leakage-style gain from total_pageviews: {BEST_AUC - HON_AUC:+.4f} AUC")
print(f"\nsame weights on FRESH: {evaluate(FRAMES['FRESH'], blend(COMP['FRESH'], BEST_W))}")

with pageviews   : AUC 0.6806  {'pv': 3.0, 'ctr': 0.5, 'age': 3.0, 'sem': 1.0, 'bm25': 0.5, 'sub': 0.5}
without pageviews: AUC 0.6733  {'pv': 0.0, 'ctr': 1.0, 'age': 3.0, 'sem': 1.0, 'bm25': 0.5, 'sub': 0.5}

searched in 290s
leakage-style gain from total_pageviews: +0.0073 AUC

same weights on FRESH: {'AUC': np.float64(0.6833888539995562), 'MRR': np.float64(0.4545805072228028), 'nDCG@5': np.float64(0.5096101272345122), 'nDCG@10': np.float64(0.5609060093368073), 'n': 60000}


# 11. A2 extension — NRMS neural news re-ranker

This section extends the A1 EB-NeRD pipeline with **NRMS (Neural News Recommendation with Multi-Head Self-Attention)**.

NRMS has three core parts:
1. **News encoder:** word embeddings → word-level multi-head self-attention → additive word attention.
2. **User encoder:** clicked-news representations → news-level multi-head self-attention → additive news attention.
3. **Click predictor:** dot product between the user and candidate-news representations.

This follows the architecture described by Wu et al. (EMNLP-IJCNLP 2019). The original paper trains with one clicked news item as a positive and non-clicked items from the same impression as negatives, using a softmax classification loss. NRMS paper: Wu et al., EMNLP-IJCNLP 2019 (ACL Anthology).

### A2 experiment

- **A1 baseline:** the tuned EB-NeRD hybrid blend from Section 9.
- **A2 improvement:** NRMS neural ranker trained on EB-NeRD click impressions.
- **Ablation:** NRMS without the user-level multi-head self-attention layer.
- **Statistics:** paired impression-level bootstrap 95% CIs.
- **Serving:** precompute all article representations once; compute each user's representation once; score candidate slates by dot product.
- **Final output:** `predictions_nrms.txt` and `ebnerd_nrms_predictions.zip`.

The implementation uses **titles**, as in the original NRMS paper. For EB-NeRD, titles are tokenized from the Danish article catalogue and the word embedding table is learned on the EB-NeRD training data. The original paper used 300-dimensional GloVe initialization; here we use a trainable 300-dimensional embedding because the EB-NeRD titles are Danish and we do not assume an English GloVe vocabulary.


In [12]:
# ============================================================
# 11.1 NRMS configuration
# ============================================================
SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Paper-inspired architecture.
WORD_DIM = 300
MODEL_DIM = 256
NUM_HEADS = 16
HEAD_DIM = MODEL_DIM // NUM_HEADS
NEWS_ATTN_DIM = 200
USER_ATTN_DIM = 200
DROPOUT = 0.20
MAX_TITLE_LEN = 30
MAX_HISTORY = 30

# Practical Kaggle settings. Increase TRAIN_IMPRESSIONS if runtime permits.
TRAIN_IMPRESSIONS = 100_000
NEGATIVES = 4
BATCH_SIZE = 64
EPOCHS = 5
LR = 1e-3
WEIGHT_DECAY = 1e-6
GRAD_CLIP = 5.0

# Same validation size as A1 for paired comparison.
NRMS_EVAL_IMPRESSIONS = 60_000
NRMS_VAL_DF = val_late.head(NRMS_EVAL_IMPRESSIONS)
# Run a second model with user self-attention removed for the A2 ablation.
RUN_NRMS_ABLATION = True

# Full test prediction.
PRED_BATCH_IMPRESSIONS = 20_000
ARTICLE_ENCODE_BATCH = 512
USER_ENCODE_BATCH = 1024

# Test output.
NRMS_OUT_TXT = SCRATCH / "predictions_nrms.txt"
NRMS_ZIP = WORK / "ebnerd_nrms_predictions.zip"

print({
    "WORD_DIM": WORD_DIM,
    "MODEL_DIM": MODEL_DIM,
    "NUM_HEADS": NUM_HEADS,
    "TRAIN_IMPRESSIONS": TRAIN_IMPRESSIONS,
    "NEGATIVES": NEGATIVES,
    "BATCH_SIZE": BATCH_SIZE,
    "EPOCHS": EPOCHS,
    "MAX_TITLE_LEN": MAX_TITLE_LEN
})


device: cuda
GPU: Tesla T4
{'WORD_DIM': 300, 'MODEL_DIM': 256, 'NUM_HEADS': 16, 'TRAIN_IMPRESSIONS': 100000, 'NEGATIVES': 4, 'BATCH_SIZE': 64, 'EPOCHS': 5, 'MAX_TITLE_LEN': 30}


## 11.2 Build a Danish title vocabulary

The original NRMS represents a news title as a sequence of word embeddings. We therefore build a vocabulary directly from EB-NeRD titles.

`PAD=0` and `UNK=1`. The vocabulary is capped to keep the trainable embedding matrix manageable.


In [13]:
# ============================================================
# Vocabulary + title tensor
# ============================================================
TOKEN_NRMS = re.compile(r"[^\W_]+", re.UNICODE)

MAX_VOCAB = 100_000
MIN_FREQ = 2

counter = Counter()
titles = ART["title"].fill_null("").to_list()

for text_ in titles:
    counter.update(TOKEN_NRMS.findall(str(text_).lower()))

vocab_items = [(w, c) for w, c in counter.items() if c >= MIN_FREQ]
vocab_items.sort(key=lambda x: (-x[1], x[0]))
vocab_items = vocab_items[:MAX_VOCAB]

WORD2ID = {"<PAD>": 0, "<UNK>": 1}
for w, _ in vocab_items:
    WORD2ID[w] = len(WORD2ID)

print("vocab size:", len(WORD2ID))
print("most common:", vocab_items[:10])

TITLE_IDS = np.zeros(
    (len(CATALOGUE), MAX_TITLE_LEN),
    dtype=np.int64
)

for i, text_ in enumerate(titles):
    toks = TOKEN_NRMS.findall(str(text_).lower())[:MAX_TITLE_LEN]
    ids = [WORD2ID.get(t, 1) for t in toks]
    if not ids:
        ids = [1]
    TITLE_IDS[i, :len(ids)] = ids

TITLE_IDS_T = torch.from_numpy(TITLE_IDS)
print("TITLE_IDS:", TITLE_IDS_T.shape)


vocab size: 28595
most common: [('i', 26885), ('på', 16077), ('er', 16049), ('og', 11612), ('for', 10653), ('til', 10460), ('med', 8994), ('det', 8053), ('af', 7747), ('fra', 7370)]
TITLE_IDS: torch.Size([125541, 30])


In [14]:
# ============================================================
# NRMS modules
# ============================================================

class AdditiveAttention(nn.Module):
    def __init__(self, dim, attn_dim):
        super().__init__()
        self.proj = nn.Linear(dim, attn_dim)
        self.query = nn.Parameter(torch.empty(attn_dim))
        self.bias = nn.Parameter(torch.zeros(attn_dim))
        nn.init.normal_(self.query, std=0.02)

    def forward(self, x, mask=None):
        # x: [B, L, D]
        h = torch.tanh(self.proj(x) + self.bias)
        score = torch.matmul(h, self.query)

        if mask is not None:
            score = score.masked_fill(~mask, -1e4)

        alpha = torch.softmax(score, dim=1)
        return torch.sum(alpha.unsqueeze(-1) * x, dim=1)


class NRMSNewsEncoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        word_dim=300,
        model_dim=256,
        heads=16,
        attn_dim=200,
        dropout=0.2
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            word_dim,
            padding_idx=0
        )

        self.input_proj = nn.Linear(
            word_dim,
            model_dim
        )

        self.self_attn = nn.MultiheadAttention(
            embed_dim=model_dim,
            num_heads=heads,
            dropout=dropout,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)

        self.word_attention = AdditiveAttention(
            model_dim,
            attn_dim
        )

    def forward(self, token_ids):
        # token_ids: [B, L]
        mask = token_ids.ne(0)

        # Avoid an all-masked sequence.
        empty = ~mask.any(dim=1)
        if empty.any():
            token_ids = token_ids.clone()
            token_ids[empty, 0] = 1
            mask = token_ids.ne(0)

        x = self.embedding(token_ids)
        x = self.input_proj(x)
        x = self.dropout(x)

        h, _ = self.self_attn(
            x, x, x,
            key_padding_mask=~mask,
            need_weights=False
        )

        h = self.dropout(h)
        return self.word_attention(h, mask)


class NRMSUserEncoder(nn.Module):
    def __init__(
        self,
        model_dim=256,
        heads=16,
        attn_dim=200,
        dropout=0.2,
        use_self_attention=True
    ):
        super().__init__()

        self.use_self_attention = use_self_attention

        if use_self_attention:
            self.self_attn = nn.MultiheadAttention(
                embed_dim=model_dim,
                num_heads=heads,
                dropout=dropout,
                batch_first=True
            )

        self.dropout = nn.Dropout(dropout)

        self.news_attention = AdditiveAttention(
            model_dim,
            attn_dim
        )

    def forward(self, history_vectors, mask):
        # history_vectors: [B, H, D]
        # mask: [B, H]
        empty = ~mask.any(dim=1)

        if empty.any():
            history_vectors = history_vectors.clone()
            mask = mask.clone()
            history_vectors[empty, 0] = 0.0
            mask[empty, 0] = True

        if self.use_self_attention:
            h, _ = self.self_attn(
                history_vectors,
                history_vectors,
                history_vectors,
                key_padding_mask=~mask,
                need_weights=False
            )
        else:
            # A2 ablation: remove news-level self-attention.
            h = history_vectors

        h = self.dropout(h)
        return self.news_attention(h, mask)


class NRMS(nn.Module):
    def __init__(
        self,
        vocab_size,
        use_user_self_attention=True
    ):
        super().__init__()

        self.news_encoder = NRMSNewsEncoder(
            vocab_size=vocab_size,
            word_dim=WORD_DIM,
            model_dim=MODEL_DIM,
            heads=NUM_HEADS,
            attn_dim=NEWS_ATTN_DIM,
            dropout=DROPOUT
        )

        self.user_encoder = NRMSUserEncoder(
            model_dim=MODEL_DIM,
            heads=NUM_HEADS,
            attn_dim=USER_ATTN_DIM,
            dropout=DROPOUT,
            use_self_attention=use_user_self_attention
        )

    def encode_news(self, token_ids):
        return self.news_encoder(token_ids)

    def encode_user(self, history_vectors, history_mask):
        return self.user_encoder(
            history_vectors,
            history_mask
        )

    def forward(
        self,
        history_tokens,
        history_mask,
        candidate_tokens
    ):
        # history_tokens: [B, H, L]
        # candidate_tokens: [B, C, L]

        B, H, L = history_tokens.shape
        C = candidate_tokens.shape[1]

        hist_flat = history_tokens.reshape(
            B * H,
            L
        )

        cand_flat = candidate_tokens.reshape(
            B * C,
            L
        )

        hist_news = self.encode_news(hist_flat)
        cand_news = self.encode_news(cand_flat)

        hist_news = hist_news.reshape(
            B, H, MODEL_DIM
        )

        cand_news = cand_news.reshape(
            B, C, MODEL_DIM
        )

        user = self.encode_user(
            hist_news,
            history_mask
        )

        scores = torch.bmm(
            cand_news,
            user.unsqueeze(-1)
        ).squeeze(-1)

        return scores


## 11.3 Convert EB-NeRD impressions into NRMS training examples

Each training example uses:
- the user's 21-day click history,
- one clicked in-view article as the positive,
- `K=4` non-clicked articles from the **same impression** as negatives.

This follows the negative-sampling formulation in the NRMS paper.


In [15]:
# ============================================================
# Training example construction
# ============================================================

ART_IDX = IDX
ART_IDS_LIST = ART_IDS

def build_nrms_examples(
    behaviors,
    histories,
    n_impressions,
    negatives,
    seed=SEED
):
    rng = np.random.default_rng(seed)

    n = min(n_impressions, behaviors.height)

    sampled = behaviors.sample(
        n=n,
        seed=seed,
        shuffle=True
    ).select([
        "user_id",
        "article_ids_inview",
        "article_ids_clicked"
    ])

    hsmall = histories.select([
        "user_id",
        "hist"
    ])

    joined = sampled.join(
        hsmall,
        on="user_id",
        how="left"
    )

    hist_out = []
    pos_out = []
    neg_out = []

    skipped = 0

    for row in joined.iter_rows(named=True):

        inview = row["article_ids_inview"] or []
        clicked = set(row["article_ids_clicked"] or [])

        positives = [
            a for a in inview
            if a in clicked and a in ART_IDX
        ]

        negatives_pool = [
            a for a in inview
            if a not in clicked and a in ART_IDX
        ]

        if not positives or len(negatives_pool) < negatives:
            skipped += 1
            continue

        # One positive per impression keeps training computationally manageable.
        pos = positives[
            int(rng.integers(len(positives)))
        ]

        if len(negatives_pool) > negatives:
            neg = rng.choice(
                negatives_pool,
                size=negatives,
                replace=False
            ).tolist()
        else:
            neg = negatives_pool

        h = [
            ART_IDX[a]
            for a in (row["hist"] or [])
            if a in ART_IDX
        ][-MAX_HISTORY:]

        # Right-align history so recent clicks are retained.
        hpad = [-1] * (MAX_HISTORY - len(h)) + h

        hist_out.append(hpad)
        pos_out.append(ART_IDX[pos])
        neg_out.append([ART_IDX[a] for a in neg])

    hist_arr = np.asarray(
        hist_out,
        dtype=np.int64
    )

    pos_arr = np.asarray(
        pos_out,
        dtype=np.int64
    )

    neg_arr = np.asarray(
        neg_out,
        dtype=np.int64
    )

    print(
        f"NRMS examples: {len(hist_arr):,} "
        f"(skipped {skipped:,})"
    )

    return hist_arr, pos_arr, neg_arr


t0 = time.time()

TRAIN_HIST_IDX, TRAIN_POS_IDX, TRAIN_NEG_IDX = build_nrms_examples(
    bh_train,
    HIST_TRAIN,
    TRAIN_IMPRESSIONS,
    NEGATIVES
)

print(
    "construction time:",
    round(time.time() - t0, 1),
    "sec"
)

print(
    "history:", TRAIN_HIST_IDX.shape,
    "positive:", TRAIN_POS_IDX.shape,
    "negative:", TRAIN_NEG_IDX.shape
)


NRMS examples: 99,971 (skipped 29)
construction time: 3.6 sec
history: (99971, 30) positive: (99971,) negative: (99971, 4)


In [16]:
# ============================================================
# Training helpers
# ============================================================

def gather_title_tensor(article_indices, device):
    safe = np.where(
        np.asarray(article_indices) < 0,
        0,
        np.asarray(article_indices)
    )
    x = TITLE_IDS_T[
        torch.as_tensor(
            safe,
            dtype=torch.long
        )
    ]
    return x.to(device)


def train_nrms(
    model,
    hist_idx,
    pos_idx,
    neg_idx,
    epochs=5,
    batch_size=64,
    lr=1e-3,
    tag="NRMS"
):
    model = model.to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=WEIGHT_DECAY
    )

    ds = TensorDataset(
        torch.from_numpy(hist_idx),
        torch.from_numpy(pos_idx),
        torch.from_numpy(neg_idx)
    )

    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    model.train()

    for epoch in range(epochs):

        total_loss = 0.0
        total_n = 0
        t0 = time.time()

        for step, (hidx, pidx, nidx) in enumerate(loader):

            hidx = hidx.numpy()
            pidx = pidx.numpy()
            nidx = nidx.numpy()

            B = len(hidx)

            # ------------------------------------------------
            # Gather title tokens
            # ------------------------------------------------
            hist_tokens = gather_title_tensor(
                hidx.reshape(-1),
                DEVICE
            ).reshape(
                B,
                MAX_HISTORY,
                MAX_TITLE_LEN
            )

            cand_idx = np.concatenate(
                [
                    pidx[:, None],
                    nidx
                ],
                axis=1
            )

            cand_tokens = gather_title_tensor(
                cand_idx.reshape(-1),
                DEVICE
            ).reshape(
                B,
                1 + NEGATIVES,
                MAX_TITLE_LEN
            )

            history_mask = (
                torch.from_numpy(
                    hidx >= 0
                )
                .to(DEVICE)
            )

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(
                enabled=torch.cuda.is_available()
            ):
                scores = model(
                    hist_tokens,
                    history_mask,
                    cand_tokens
                )

                # Positive candidate is always index 0.
                target = torch.zeros(
                    B,
                    dtype=torch.long,
                    device=DEVICE
                )

                loss = F.cross_entropy(
                    scores,
                    target
                )

            if torch.isnan(loss):
                raise RuntimeError(
                    f"{tag}: NaN loss at step {step}"
                )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP
            )

            optimizer.step()

            total_loss += (
                float(loss.detach()) * B
            )
            total_n += B

            if step % 250 == 0:
                print(
                    f"{tag} epoch {epoch+1}/{epochs} "
                    f"step {step:,}/{len(loader):,} "
                    f"loss {total_loss/max(total_n,1):.4f}"
                )

        print(
            f"{tag} epoch {epoch+1}: "
            f"loss={total_loss/max(total_n,1):.4f} "
            f"time={time.time()-t0:.1f}s"
        )

    return model


print("training helper ready")


training helper ready


In [17]:
# ============================================================
# Train the full NRMS model
# ============================================================

NRMS_FULL = NRMS(
    vocab_size=len(WORD2ID),
    use_user_self_attention=True
)

print(
    "NRMS parameters:",
    sum(p.numel() for p in NRMS_FULL.parameters()) / 1e6,
    "M"
)

NRMS_FULL = train_nrms(
    NRMS_FULL,
    TRAIN_HIST_IDX,
    TRAIN_POS_IDX,
    TRAIN_NEG_IDX,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    tag="NRMS-full"
)

torch.cuda.empty_cache()
gc.collect()


NRMS parameters: 9.285492 M
NRMS-full epoch 1/5 step 0/1,563 loss 1.6067
NRMS-full epoch 1/5 step 250/1,563 loss 1.5506
NRMS-full epoch 1/5 step 500/1,563 loss 1.5198
NRMS-full epoch 1/5 step 750/1,563 loss 1.5054
NRMS-full epoch 1/5 step 1,000/1,563 loss 1.4939
NRMS-full epoch 1/5 step 1,250/1,563 loss 1.4843
NRMS-full epoch 1/5 step 1,500/1,563 loss 1.4778
NRMS-full epoch 1: loss=1.4759 time=85.6s
NRMS-full epoch 2/5 step 0/1,563 loss 1.4727
NRMS-full epoch 2/5 step 250/1,563 loss 1.4288
NRMS-full epoch 2/5 step 500/1,563 loss 1.4298
NRMS-full epoch 2/5 step 750/1,563 loss 1.4282
NRMS-full epoch 2/5 step 1,000/1,563 loss 1.4261
NRMS-full epoch 2/5 step 1,250/1,563 loss 1.4235
NRMS-full epoch 2/5 step 1,500/1,563 loss 1.4230
NRMS-full epoch 2: loss=1.4226 time=89.5s
NRMS-full epoch 3/5 step 0/1,563 loss 1.4498
NRMS-full epoch 3/5 step 250/1,563 loss 1.4016
NRMS-full epoch 3/5 step 500/1,563 loss 1.4042
NRMS-full epoch 3/5 step 750/1,563 loss 1.4043
NRMS-full epoch 3/5 step 1,000/1,563

0

## 11.4 Precompute article representations

NRMS separates the news encoder from the user encoder. Therefore the learned representation of every article can be computed once after training.

This makes large-scale inference much cheaper:
- encode 125k articles once;
- encode each validation/test user once;
- score candidate articles by a dot product.


In [18]:
# ============================================================
# Encode the complete EB-NeRD article catalogue
# ============================================================

@torch.no_grad()
def encode_all_articles(model):
    model.eval()

    outs = []

    for st in range(
        0,
        len(CATALOGUE),
        ARTICLE_ENCODE_BATCH
    ):

        ids = np.arange(
            st,
            min(
                st + ARTICLE_ENCODE_BATCH,
                len(CATALOGUE)
            )
        )

        tokens = gather_title_tensor(
            ids,
            DEVICE
        )

        z = model.encode_news(
            tokens
        )

        z = F.normalize(
            z,
            dim=-1
        )

        outs.append(
            z.cpu().numpy().astype(
                np.float32
            )
        )

    return np.concatenate(
        outs,
        axis=0
    )


t0 = time.time()

NRMS_ARTICLE_VEC = encode_all_articles(
    NRMS_FULL
)

print(
    "article vectors:",
    NRMS_ARTICLE_VEC.shape,
    "time:",
    round(time.time()-t0, 1),
    "sec"
)

print(
    "memory:",
    round(
        NRMS_ARTICLE_VEC.nbytes / 1e6,
        1
    ),
    "MB"
)


article vectors: (125541, 256) time: 2.1 sec
memory: 128.6 MB


In [19]:
# ============================================================
# User representation from a history table
# ============================================================

def history_to_indices(hist_list):
    out = [ART_IDX[a] for a in hist_list if a in ART_IDX]
    return out[-MAX_HISTORY:]


@torch.no_grad()
def encode_users_from_history(
    history_df,
    article_vec,
    batch_size=USER_ENCODE_BATCH
):
    model = NRMS_FULL
    model.eval()

    users = history_df["user_id"].to_numpy()
    histories = history_df["hist"].to_list()

    user_vec = np.zeros(
        (len(users), MODEL_DIM),
        dtype=np.float32
    )

    for st in range(
        0,
        len(users),
        batch_size
    ):

        hs = histories[
            st:st + batch_size
        ]

        idx = np.zeros(
            (
                len(hs),
                MAX_HISTORY
            ),
            dtype=np.int64
        )

        mask = np.zeros(
            (
                len(hs),
                MAX_HISTORY
            ),
            dtype=bool
        )

        for r, h in enumerate(hs):

            hidx = history_to_indices(
                h or []
            )

            if hidx:
                hidx = hidx[-MAX_HISTORY:]
                idx[
                    r,
                    -len(hidx):
                ] = hidx

                mask[
                    r,
                    -len(hidx):
                ] = True

        hv = torch.from_numpy(
            article_vec[idx]
        ).to(DEVICE)

        hm = torch.from_numpy(
            mask
        ).to(DEVICE)

        u = model.encode_user(
            hv,
            hm
        )

        u = F.normalize(
            u,
            dim=-1
        )

        user_vec[
            st:st + len(hs)
        ] = u.cpu().numpy().astype(
            np.float32
        )

    return users, user_vec


print("user encoder helper ready")


user encoder helper ready


In [20]:
# ============================================================
# Validation user vectors
# ============================================================

t0 = time.time()

VAL_USERS, VAL_USER_VEC = encode_users_from_history(
    HIST_EVAL,
    NRMS_ARTICLE_VEC
)

VAL_UID_TO_ROW = {
    int(u): i
    for i, u in enumerate(VAL_USERS)
}

print(
    "validation users:",
    len(VAL_USERS),
    "time:",
    round(time.time()-t0, 1),
    "sec"
)


validation users: 791582 time: 23.0 sec


## 11.4 (cont.) — Aligned NRMS validation scoring

Scores every candidate in `val_late` in impression order, assigning `score=0.0` to
articles absent from the vocabulary. This guarantees the same candidate set as A1 and
makes impression-level paired comparison valid.
The NRMS score vector and the A1 score vector must cover identical (impression, article)
pairs so the paired bootstrap is meaningful.

In [21]:
# ============================================================
# Aligned NRMS validation scorer
# Produces (imp_ids, labels, scores) with one entry per (impression, in-view article)
# in the order val_late iterates, so A1 and NRMS are directly comparable.
# ============================================================

def nrms_scores_aligned(df, user_vec, uid_to_row, article_vec):
    """
    For every (impression, in-view article) pair in df, return:
      imp_ids : impression id (repeated per candidate)
      labels  : 1 if article was clicked, else 0
      scores  : NRMS dot-product score (0.0 for articles not in ART_IDX)
    Article order within each impression matches the original slate.
    """
    scores, labels, imp_ids = [], [], []
    for r in df.iter_rows(named=True):
        uid = int(r["user_id"])
        ui  = uid_to_row.get(uid)
        u   = user_vec[ui] if ui is not None else np.zeros(MODEL_DIM, dtype=np.float32)
        inview  = r["article_ids_inview"]  or []
        clicked = set(r["article_ids_clicked"] or [])
        for a in inview:
            s = float(article_vec[ART_IDX[a]] @ u) if a in ART_IDX else 0.0
            scores.append(s)
            labels.append(int(a in clicked))
            imp_ids.append(r["impression_id"])
    return (
        np.asarray(imp_ids),
        np.asarray(labels, dtype=np.int8),
        np.asarray(scores, dtype=np.float32),
    )


t0 = time.time()
NRMS_IMP_IDS, NRMS_LABELS, NRMS_VAL_SCORE = nrms_scores_aligned(
    NRMS_VAL_DF, VAL_USER_VEC, VAL_UID_TO_ROW, NRMS_ARTICLE_VEC
)
print(f"aligned NRMS scoring: {len(NRMS_VAL_SCORE):,} rows in {time.time()-t0:.1f}s")

NRMS_VAL_METRICS = score_fast(NRMS_IMP_IDS, NRMS_LABELS, NRMS_VAL_SCORE)

# A1 score on the SAME candidate rows in the SAME impression order
# NOTE: FRAMES['STALE'] was built with prepare() which may exclude articles not in the
# catalogue, so we rebuild A1 scores in the aligned iteration order here rather than
# relying on FRAMES['STALE']['imp'] == NRMS_IMP_IDS (they may differ).
A1_ALIGNED_SCORE = []
for r in NRMS_VAL_DF.iter_rows(named=True):
    inview = r["article_ids_inview"] or []
    for a in inview:
        if a in ART_IDX:
            ci = ART_IDX[a]
            ts = int(np.datetime64(r["impression_time"], 's').astype(np.int64))
            have_age = A_PUB_OK[ci]
            age = float(max((ts - A_PUB[ci]) / 3600.0, 0.0)) if have_age else 0.0
            age_s = float(AGE_VALUE[np.searchsorted(AGE_EDGES, age)])
            pv_s  = float(A_LOGPV[ci])
            ctr_s = float(CTRV_STALE[ci])
            A1_ALIGNED_SCORE.append(
                BEST_W.get('pv', 0) * pv_s +
                BEST_W.get('ctr', 0) * ctr_s +
                BEST_W.get('age', 0) * age_s
            )
        else:
            A1_ALIGNED_SCORE.append(0.0)
A1_ALIGNED_SCORE = np.asarray(A1_ALIGNED_SCORE, dtype=np.float32)

assert len(A1_ALIGNED_SCORE) == len(NRMS_VAL_SCORE), \
    f"score length mismatch: A1={len(A1_ALIGNED_SCORE)} NRMS={len(NRMS_VAL_SCORE)}"

A1_VAL_METRICS = score_fast(NRMS_IMP_IDS, NRMS_LABELS, A1_ALIGNED_SCORE)

print("A1  (aligned):", A1_VAL_METRICS)
print("NRMS          :", NRMS_VAL_METRICS)


aligned NRMS scoring: 736,204 rows in 1.8s
A1  (aligned): {'AUC': np.float64(0.6176999263770006), 'MRR': np.float64(0.3951900385302369), 'nDCG@5': np.float64(0.4414709485239739), 'nDCG@10': np.float64(0.5102615979018666), 'n': 60000}
NRMS          : {'AUC': np.float64(0.5532882449851358), 'MRR': np.float64(0.3412556648372597), 'nDCG@5': np.float64(0.3822510655092956), 'nDCG@10': np.float64(0.45850860246667136), 'n': 60000}


## 11.5 A2 ablation — remove user-level self-attention

The full NRMS user encoder models interactions among clicked news with multi-head self-attention.

For the ablation, we remove this layer and keep only additive attention over the clicked-news representations.

This isolates the contribution of the **news-level self-attention** component.


In [22]:
# ============================================================
# Train NRMS ablation
# ============================================================

NRMS_ABLATION = None

if RUN_NRMS_ABLATION:

    NRMS_ABLATION = NRMS(
        vocab_size=len(WORD2ID),
        use_user_self_attention=False
    )

    print(
        "Ablation parameters:",
        sum(p.numel() for p in NRMS_ABLATION.parameters()) / 1e6,
        "M"
    )

    NRMS_ABLATION = train_nrms(
        NRMS_ABLATION,
        TRAIN_HIST_IDX,
        TRAIN_POS_IDX,
        TRAIN_NEG_IDX,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        tag="NRMS-no-user-self-attn"
    )

    torch.cuda.empty_cache()
    gc.collect()


Ablation parameters: 9.022324 M
NRMS-no-user-self-attn epoch 1/5 step 0/1,563 loss 1.6160
NRMS-no-user-self-attn epoch 1/5 step 250/1,563 loss 1.5784
NRMS-no-user-self-attn epoch 1/5 step 500/1,563 loss 1.5429
NRMS-no-user-self-attn epoch 1/5 step 750/1,563 loss 1.5214
NRMS-no-user-self-attn epoch 1/5 step 1,000/1,563 loss 1.5060
NRMS-no-user-self-attn epoch 1/5 step 1,250/1,563 loss 1.4933
NRMS-no-user-self-attn epoch 1/5 step 1,500/1,563 loss 1.4844
NRMS-no-user-self-attn epoch 1: loss=1.4832 time=87.1s
NRMS-no-user-self-attn epoch 2/5 step 0/1,563 loss 1.3861
NRMS-no-user-self-attn epoch 2/5 step 250/1,563 loss 1.4216
NRMS-no-user-self-attn epoch 2/5 step 500/1,563 loss 1.4234
NRMS-no-user-self-attn epoch 2/5 step 750/1,563 loss 1.4215
NRMS-no-user-self-attn epoch 2/5 step 1,000/1,563 loss 1.4196
NRMS-no-user-self-attn epoch 2/5 step 1,250/1,563 loss 1.4175
NRMS-no-user-self-attn epoch 2/5 step 1,500/1,563 loss 1.4159
NRMS-no-user-self-attn epoch 2: loss=1.4155 time=86.9s
NRMS-no-us

### Evaluate ablation on validation

Score NRMS-no-user-self-attn on the same 60k impressions using the same aligned scorer.

In [23]:
# ============================================================
# Evaluate ablation model on validation (same aligned impressions)
# ============================================================

ABLATION_VAL_METRICS = None
ABL_VAL_SCORE = None

if NRMS_ABLATION is not None:
    # Encode articles with the ablation model
    NRMS_ABL_ARTICLE_VEC = encode_all_articles(NRMS_ABLATION)

    # Encode validation users with ablation model
    ABL_VAL_USERS, ABL_VAL_USER_VEC = encode_users_from_history(
        HIST_EVAL, NRMS_ABL_ARTICLE_VEC
    )
    ABL_VAL_UID_TO_ROW = {int(u): i for i, u in enumerate(ABL_VAL_USERS)}

    # Score validation impressions
    _, _, ABL_VAL_SCORE = nrms_scores_aligned(
        NRMS_VAL_DF, ABL_VAL_USER_VEC, ABL_VAL_UID_TO_ROW, NRMS_ABL_ARTICLE_VEC
    )
    ABLATION_VAL_METRICS = score_fast(NRMS_IMP_IDS, NRMS_LABELS, ABL_VAL_SCORE)

    torch.cuda.empty_cache()
    gc.collect()

    print("A1  (aligned)       :", A1_VAL_METRICS)
    print("NRMS (full)         :", NRMS_VAL_METRICS)
    print("NRMS-no-user-attn   :", ABLATION_VAL_METRICS)
else:
    print("RUN_NRMS_ABLATION=False, ablation skipped")


A1  (aligned)       : {'AUC': np.float64(0.6176999263770006), 'MRR': np.float64(0.3951900385302369), 'nDCG@5': np.float64(0.4414709485239739), 'nDCG@10': np.float64(0.5102615979018666), 'n': 60000}
NRMS (full)         : {'AUC': np.float64(0.5532882449851358), 'MRR': np.float64(0.3412556648372597), 'nDCG@5': np.float64(0.3822510655092956), 'nDCG@10': np.float64(0.45850860246667136), 'n': 60000}
NRMS-no-user-attn   : {'AUC': np.float64(0.5111447840196459), 'MRR': np.float64(0.30884851508508077), 'nDCG@5': np.float64(0.3455162036307104), 'nDCG@10': np.float64(0.4276414317940881), 'n': 60000}


In [24]:
# ============================================================
# Paired bootstrap: NRMS vs A1 (and ablation vs NRMS if available)
# ============================================================

def metric_per_impression(imp_ids, labels, scores):
    """Per-impression metric array: shape (n_impressions, 4) = [AUC, MRR, nDCG@5, nDCG@10]"""
    unique_ids = np.unique(imp_ids)
    out = []
    for imp in unique_ids:
        idx = np.flatnonzero(imp_ids == imp)
        y = labels[idx].astype(np.float64)
        s = scores[idx].astype(np.float64)
        if y.sum() == 0 or y.sum() == len(y):
            continue  # skip degenerate impressions
        m = score_fast(
            np.zeros(len(idx), dtype=np.int64),
            labels[idx], scores[idx]
        )
        out.append([m["AUC"], m["MRR"], m["nDCG@5"], m["nDCG@10"]])
    return np.asarray(out, dtype=np.float64)


def paired_bootstrap(imp_ids, labels, score_a, score_b, reps=1000, seed=2026):
    """Paired bootstrap CI for (model_b - model_a) on AUC, MRR, nDCG@5, nDCG@10."""
    per_a = metric_per_impression(imp_ids, labels, score_a)
    per_b = metric_per_impression(imp_ids, labels, score_b)
    n = min(len(per_a), len(per_b))
    per_a, per_b = per_a[:n], per_b[:n]
    delta = per_b - per_a
    rng = np.random.default_rng(seed)
    boot = np.empty((reps, delta.shape[1]), dtype=np.float64)
    for r in range(reps):
        idx = rng.integers(0, n, size=n)
        boot[r] = delta[idx].mean(axis=0)
    point = delta.mean(axis=0)
    lo = np.quantile(boot, 0.025, axis=0)
    hi = np.quantile(boot, 0.975, axis=0)
    return pl.DataFrame({
        "metric": ["AUC", "MRR", "nDCG@5", "nDCG@10"],
        "delta_b_minus_a": point,
        "ci_low": lo, "ci_high": hi,
        "excludes_zero": ((lo > 0) | (hi < 0)),
    })


print("=== Paired bootstrap: NRMS vs A1 ===")
BOOT_NRMS_VS_A1 = paired_bootstrap(
    NRMS_IMP_IDS, NRMS_LABELS, A1_ALIGNED_SCORE, NRMS_VAL_SCORE, reps=1000, seed=2026
)
print(BOOT_NRMS_VS_A1)

BOOT_ABL_VS_NRMS = None
if ABL_VAL_SCORE is not None:
    print("\n=== Paired bootstrap: NRMS-no-user-attn vs NRMS-full ===")
    BOOT_ABL_VS_NRMS = paired_bootstrap(
        NRMS_IMP_IDS, NRMS_LABELS, NRMS_VAL_SCORE, ABL_VAL_SCORE, reps=1000, seed=2026
    )
    print(BOOT_ABL_VS_NRMS)


=== Paired bootstrap: NRMS vs A1 ===
shape: (4, 5)
┌─────────┬─────────────────┬───────────┬───────────┬───────────────┐
│ metric  ┆ delta_b_minus_a ┆ ci_low    ┆ ci_high   ┆ excludes_zero │
│ ---     ┆ ---             ┆ ---       ┆ ---       ┆ ---           │
│ str     ┆ f64             ┆ f64       ┆ f64       ┆ bool          │
╞═════════╪═════════════════╪═══════════╪═══════════╪═══════════════╡
│ AUC     ┆ -0.064412       ┆ -0.067864 ┆ -0.06106  ┆ true          │
│ MRR     ┆ -0.053934       ┆ -0.057097 ┆ -0.050847 ┆ true          │
│ nDCG@5  ┆ -0.05922        ┆ -0.062639 ┆ -0.055923 ┆ true          │
│ nDCG@10 ┆ -0.051753       ┆ -0.054563 ┆ -0.049064 ┆ true          │
└─────────┴─────────────────┴───────────┴───────────┴───────────────┘

=== Paired bootstrap: NRMS-no-user-attn vs NRMS-full ===
shape: (4, 5)
┌─────────┬─────────────────┬───────────┬───────────┬───────────────┐
│ metric  ┆ delta_b_minus_a ┆ ci_low    ┆ ci_high   ┆ excludes_zero │
│ ---     ┆ ---             ┆ ---    

## 11.6 Final fallback-aware NRMS scoring

For test submission we cannot drop candidates. Every in-view article must receive a rank.

Therefore:
- known EB-NeRD articles receive the NRMS score;
- unseen/unsupported articles receive the A1 score;
- ties are broken by original slate position.

This guarantees a complete `1..N` permutation for every impression.


In [25]:
# ============================================================
# Test user vectors
# ============================================================

t0 = time.time()

TEST_USERS, TEST_USER_VEC = encode_users_from_history(
    HIST_PRED,
    NRMS_ARTICLE_VEC
)

TEST_UID_TO_ROW = {
    int(u): i
    for i, u in enumerate(TEST_USERS)
}

print(
    "test users:",
    len(TEST_USERS),
    "time:",
    round(time.time()-t0, 1),
    "sec"
)


test users: 807677 time: 24.6 sec


In [26]:
# ============================================================
# Final test prediction
# ============================================================

def a1_fallback_scores_for_test(df_slice):
    F = prepare(
        df_slice,
        CTRV_FINAL,
        False
    )
    C = components(
        F,
        UID_PRED,
        H_PRED
    )
    return F, blend(C, BEST_W)


def nrms_rank_test_batch(df_slice):
    F = prepare(
        df_slice,
        CTRV_FINAL,
        False
    )

    n = F["n"]

    ui, u_ok = lookup(
        TEST_USERS,
        F["users"]
    )

    nrms_score = np.full(
        n,
        -np.inf,
        dtype=np.float32
    )

    valid = F["c_ok"] & u_ok

    if np.any(valid):
        uvec = TEST_USER_VEC[ui[valid]]
        cidx = F["ci"][valid]

        nrms_score[valid] = np.einsum(
            "ij,ij->i",
            NRMS_ARTICLE_VEC[cidx],
            uvec
        )

    missing = ~np.isfinite(nrms_score)

    if np.any(missing):
        F2, a1score = a1_fallback_scores_for_test(
            df_slice
        )
        nrms_score[missing] = a1score[missing]
        del F2, a1score

    return F, nrms_score


# Batch by candidate rows.
_len = bh_test[
    "article_ids_inview"
].list.len().to_numpy().astype(np.int64)

_grp = (
    np.cumsum(_len) - 1
) // 300_000

STARTS = np.flatnonzero(
    np.diff(
        _grp,
        prepend=np.int64(-1)
    )
)

SIZES = np.diff(
    np.append(
        STARTS,
        len(_len)
    )
)

print(
    f"{len(STARTS):,} batches | "
    f"max impressions/batch {int(SIZES.max()):,}"
)

if NRMS_OUT_TXT.exists():
    NRMS_OUT_TXT.unlink()

t0 = time.time()
written = 0

with open(
    NRMS_OUT_TXT,
    "w",
    encoding="utf-8",
    newline="\n"
) as fh:

    for bi, (start, size) in enumerate(
        zip(STARTS, SIZES)
    ):

        batch = bh_test.slice(
            int(start),
            int(size)
        )

        F, score = nrms_rank_test_batch(
            batch
        )

        ranks = segment_ranks(
            score,
            F["lens"]
        )

        out = []
        p = 0

        for iid, k in zip(
            F["imp_ids"].tolist(),
            F["lens"].tolist()
        ):
            out.append(
                f"{iid} "
                f"[{','.join(map(str, ranks[p:p+k].tolist()))}]"
            )
            p += k

        fh.write(
            "\n".join(out) + "\n"
        )

        written += len(out)

        del batch, F, score, ranks, out

        if bi % 25 == 0:
            gc.collect()

            frac = written / bh_test.height

            print(
                f"{written:,}/{bh_test.height:,} "
                f"({100*frac:.1f}%) | "
                f"elapsed {time.time()-t0:.0f}s"
            )

print(
    f"\nNRMS prediction written: {written:,} lines"
)

print(
    "size:",
    round(
        NRMS_OUT_TXT.stat().st_size / 1e6,
        1
    ),
    "MB"
)


687 batches | max impressions/batch 27,116
25,603/13,536,710 (0.2%) | elapsed 2s
667,728/13,536,710 (4.9%) | elapsed 54s
1,309,369/13,536,710 (9.7%) | elapsed 106s
1,950,118/13,536,710 (14.4%) | elapsed 159s
2,591,203/13,536,710 (19.1%) | elapsed 211s
3,230,359/13,536,710 (23.9%) | elapsed 263s
3,876,065/13,536,710 (28.6%) | elapsed 315s
4,519,755/13,536,710 (33.4%) | elapsed 367s
5,162,347/13,536,710 (38.1%) | elapsed 420s
5,808,084/13,536,710 (42.9%) | elapsed 472s
6,448,312/13,536,710 (47.6%) | elapsed 524s
7,090,028/13,536,710 (52.4%) | elapsed 577s
7,729,887/13,536,710 (57.1%) | elapsed 629s
8,365,243/13,536,710 (61.8%) | elapsed 682s
9,007,015/13,536,710 (66.5%) | elapsed 734s
9,648,563/13,536,710 (71.3%) | elapsed 787s
10,294,810/13,536,710 (76.1%) | elapsed 839s
10,936,817/13,536,710 (80.8%) | elapsed 892s
11,575,924/13,536,710 (85.5%) | elapsed 944s
12,214,570/13,536,710 (90.2%) | elapsed 996s
12,851,575/13,536,710 (94.9%) | elapsed 1049s
13,344,206/13,536,710 (98.6%) | elapse

In [27]:
# ============================================================
# Validate NRMS submission
# ============================================================

def parse_line_nrms(line):
    impid, ranks = line.strip("\n").split()
    return impid, json.loads(ranks)


expected_ids = bh_test[
    "impression_id"
].to_list()

expected_len = bh_test[
    "article_ids_inview"
].list.len().to_list()

problems = []
n = 0

with open(
    NRMS_OUT_TXT,
    encoding="utf-8"
) as fh:

    for i, line in enumerate(fh):

        n += 1

        try:
            impid, ranks = parse_line_nrms(
                line
            )
        except Exception as e:
            problems.append(
                f"line {i+1}: unparseable ({e})"
            )
            continue

        if impid != str(
            expected_ids[i]
        ):
            problems.append(
                f"line {i+1}: wrong impression id"
            )

        if len(ranks) != expected_len[i]:
            problems.append(
                f"line {i+1}: {len(ranks)} ranks "
                f"for slate {expected_len[i]}"
            )

        if sorted(ranks) != list(
            range(1, len(ranks)+1)
        ):
            problems.append(
                f"line {i+1}: not a permutation"
            )

        if len(problems) > 10:
            break

if n != len(expected_ids):
    problems.append(
        f"{n:,} lines but "
        f"{len(expected_ids):,} impressions"
    )

print("lines:", f"{n:,}")

if problems:
    print("PROBLEMS — do not submit:")
    for p in problems[:10]:
        print(" ", p)
else:
    print(
        "PASSED — valid EB-NeRD NRMS prediction file"
    )

print("\nfirst 3 lines:")
with open(NRMS_OUT_TXT) as fh:
    for _ in range(3):
        print(" ", fh.readline().rstrip()[:120])


lines: 13,536,710
PASSED — valid EB-NeRD NRMS prediction file

first 3 lines:
  6451339 [9,3,2,6,8,5,4,1,7]
  6451363 [2,4,3,8,5,1,7,6]
  6451382 [5,3,1,2,4]


In [28]:
# ============================================================
# Zip for Codabench
# ============================================================

if NRMS_ZIP.exists():
    NRMS_ZIP.unlink()

with zipfile.ZipFile(
    NRMS_ZIP,
    "w",
    zipfile.ZIP_DEFLATED,
    compresslevel=6
) as z:
    z.write(
        NRMS_OUT_TXT,
        arcname="predictions.txt"
    )

with zipfile.ZipFile(NRMS_ZIP) as z:
    names = z.namelist()
    print("archive contents:", names)
    assert names == [
        "predictions.txt"
    ], "archive must contain exactly predictions.txt"

print(
    f"\n{NRMS_ZIP} "
    f"({NRMS_ZIP.stat().st_size/1e6:.1f} MB)"
)


archive contents: ['predictions.txt']

/kaggle/working/ebnerd_nrms_predictions.zip (229.9 MB)


## 11.7 A2 report artifacts

For the final design note, record:

1. **A1 baseline:** the tuned hybrid model from Section 9.
2. **A2 model:** full NRMS.
3. **Ablation:** NRMS without user-level self-attention.
4. **Metrics:** AUC, MRR, nDCG@5, nDCG@10.
5. **Paired bootstrap:** NRMS − A1 with 95% CI.
6. **Serving:** article encoding time, user encoding time, candidate scoring latency, p50/p95/p99.
7. **Scale:** number of articles, users, impressions and candidate rows.
8. **Leaderboard:** upload `ebnerd_nrms_predictions.zip` to Codabench and record the actual score.

Do **not** claim statistical significance unless the paired confidence interval excludes zero.

The original NRMS paper reports that multi-head self-attention is used at both word and news levels, with additive attention selecting important words/news, and uses a dot-product click predictor.


In [29]:
# ============================================================
# Save A2 report artifacts
# ============================================================

REPORT_DIR = WORK / "A2_NRMS_report_artifacts"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# 1. A1 vs NRMS comparison
comparison_rows = [
    {"model": "A1_hybrid", **{k: A1_VAL_METRICS[k] for k in ["AUC","MRR","nDCG@5","nDCG@10"]}},
    {"model": "NRMS_full", **{k: NRMS_VAL_METRICS[k] for k in ["AUC","MRR","nDCG@5","nDCG@10"]}},
]
if ABLATION_VAL_METRICS is not None:
    comparison_rows.append(
        {"model": "NRMS_no_user_self_attn",
         **{k: ABLATION_VAL_METRICS[k] for k in ["AUC","MRR","nDCG@5","nDCG@10"]}}
    )
pl.DataFrame(comparison_rows).write_csv(REPORT_DIR / "model_comparison.csv")

# 2. Paired bootstrap CI
BOOT_NRMS_VS_A1.write_csv(REPORT_DIR / "paired_bootstrap_nrms_vs_a1.csv")
if BOOT_ABL_VS_NRMS is not None:
    BOOT_ABL_VS_NRMS.write_csv(REPORT_DIR / "paired_bootstrap_ablation_vs_nrms.csv")

# 3. Serving latency (populated by 11.8)
# 4. Experiment summary
artifact_summary = {
    "method": "NRMS adapted to EB-NeRD",
    "baseline": "A1 tuned hybrid",
    "news_encoder": "word embedding + multi-head self-attention + additive attention",
    "user_encoder": "multi-head self-attention + additive attention",
    "click_predictor": "dot product",
    "negative_sampling": int(NEGATIVES),
    "train_impressions": int(TRAIN_IMPRESSIONS),
    "validation_impressions": int(NRMS_EVAL_IMPRESSIONS),
    "word_dim": int(WORD_DIM), "model_dim": int(MODEL_DIM),
    "heads": int(NUM_HEADS), "max_title_len": int(MAX_TITLE_LEN),
    "max_history": int(MAX_HISTORY), "epochs": int(EPOCHS),
    "device": str(DEVICE),
    "val_AUC_A1": round(A1_VAL_METRICS["AUC"], 6),
    "val_AUC_NRMS": round(NRMS_VAL_METRICS["AUC"], 6),
    "A2_submission_zip": str(NRMS_ZIP),
    "A1_submission_zip": str(A1_ZIP),
}
with open(REPORT_DIR / "experiment_summary.json", "w") as f:
    json.dump(artifact_summary, f, indent=2)

print("A2 artifacts written to:", REPORT_DIR)
for p in sorted(REPORT_DIR.rglob("*")):
    if p.is_file(): print(f"   {p.name}")


NameError: name 'A1_ZIP' is not defined

## 11.8 Serving / scale analysis

NRMS is particularly suitable for offline/news-recommendation serving because article
representations can be precomputed and user representations can be computed once per
user/session. Candidate scoring is then a vector dot product.

We benchmark:
- article encoding,
- user encoding,
- candidate scoring,
- p50/p95/p99 latency,
- approximate QPS,
- memory footprint.


In [ ]:
# ============================================================
# Serving benchmark
# ============================================================

def percentile_ms(values, q):
    return float(np.percentile(values, q))


@torch.no_grad()
def benchmark_nrms_serving(
    n_requests=100
):
    model = NRMS_FULL.eval()

    rng = np.random.default_rng(2026)

    article_times = []
    user_times = []
    score_times = []

    n_articles = len(CATALOGUE)
    n_users = len(TEST_USERS)

    for _ in range(n_requests):

        # One user and 15 candidates approximates the observed EB-NeRD slate size.
        ui = int(rng.integers(n_users))
        candidates = rng.integers(
            0,
            n_articles,
            size=16
        )

        # Article scoring.
        t0 = time.perf_counter()

        cv = torch.from_numpy(
            NRMS_ARTICLE_VEC[candidates]
        ).to(DEVICE)

        article_times.append(
            (time.perf_counter() - t0) * 1000
        )

        # User representation is already precomputed in production.
        t0 = time.perf_counter()

        uv = torch.from_numpy(
            TEST_USER_VEC[ui]
        ).to(DEVICE)

        user_times.append(
            (time.perf_counter() - t0) * 1000
        )

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        t0 = time.perf_counter()

        _ = cv @ uv

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        score_times.append(
            (time.perf_counter() - t0) * 1000
        )

    result = pl.DataFrame({
        "stage": [
            "candidate_vector_transfer",
            "user_vector_transfer",
            "candidate_scoring"
        ],
        "p50_ms": [
            percentile_ms(article_times, 50),
            percentile_ms(user_times, 50),
            percentile_ms(score_times, 50)
        ],
        "p95_ms": [
            percentile_ms(article_times, 95),
            percentile_ms(user_times, 95),
            percentile_ms(score_times, 95)
        ],
        "p99_ms": [
            percentile_ms(article_times, 99),
            percentile_ms(user_times, 99),
            percentile_ms(score_times, 99)
        ]
    })

    return result


SERVING = benchmark_nrms_serving(
    100
)

print(SERVING)
print(
    "\nApprox candidate-scoring QPS:",
    round(
        1000 / max(
            SERVING["p50_ms"][2],
            1e-6
        ),
        1
    )
)


In [ ]:
# ============================================================
# Save serving latency to report dir and print compact summary
# ============================================================

SERVING.write_csv(REPORT_DIR / "serving_latency.csv")

summary = {
    "method": "NRMS adapted to EB-NeRD",
    "baseline": "A1 tuned hybrid",
    "news_encoder": "word embedding + multi-head self-attention + additive attention",
    "user_encoder": "multi-head self-attention + additive attention",
    "click_predictor": "dot product",
    "negative_sampling": NEGATIVES,
    "train_impressions": TRAIN_IMPRESSIONS,
    "validation_impressions": NRMS_EVAL_IMPRESSIONS,
    "word_dim": WORD_DIM, "model_dim": MODEL_DIM,
    "heads": NUM_HEADS, "max_title_len": MAX_TITLE_LEN,
    "max_history": MAX_HISTORY, "epochs": EPOCHS,
    "device": str(DEVICE),
    "A1_zip": str(A1_ZIP),
    "NRMS_zip": str(NRMS_ZIP),
}
print(json.dumps(summary, indent=2))
print("\nAll report artifacts:")
for p in sorted(REPORT_DIR.rglob("*")):
    if p.is_file(): print(f"   {p.name}  ({p.stat().st_size/1e3:.1f} KB)")
